# Hands-on 20: Menyiapkan Dataset Klasifikasi Normal vs Abnormal

Buku Referensi: *IoT Terintegrasi dengan Machine Learning & Data Analytics* (Onno W. Purbo), Bab 7.

Target biner `target_abnormal` dari `is_anomaly`, fitur waktu + lag + rolling + diff,
split time-aware 70/15/15. Output: `data/processed/classification_room/`
(train/val/test + X/y terpisah).

## Tujuan

Pada latihan ini, pembaca akan menyiapkan dataset klasifikasi sederhana untuk masalah normal vs abnormal pada konteks IoT. Setelah selesai, pembaca diharapkan mampu:

- memilih target klasifikasi yang jelas,
- menyiapkan fitur dari data sensor,
- membuat label normal dan abnormal,
- melakukan split time-aware,
- dan menyimpan dataset train/validation/test secara rapi.

Latihan ini sangat penting karena di banyak use case IoT, keputusan awal yang dibutuhkan bukan prediksi yang rumit, melainkan apakah kondisi masih normal atau perlu perhatian. Referensi juga menegaskan bahwa classification adalah salah satu fondasi ML untuk IoT analytics, dan label target bisa berupa kelas nominal seperti dua kelas atau multi-kelas.

## 2. Contoh implementasi nyata

Smart room monitoring

Fitur: ●    suhu ●    kelembaban ●    CO₂ ●    daya

Target: ● normal bila kondisi masih wajar ● abnormal bila salah satu indikator melewati pola aman

Smart energy

Fitur: ●    daya ●    jam ●    rata-rata beban lokal ●    perubahan daya

Target: ● normal vs abnormal untuk beban tidak wajar

Predictive maintenance sederhana

Fitur: ●    RMS getaran ●    standar deviasi getaran ●    arus ●    suhu motor

Target: ● normal vs abnormal

## 5. Strategi yang dipakai

Kita akan:

- membaca dataset,
- mengonversi timestamp,
- membuat fitur sederhana,
- menentukan target biner normal vs abnormal,
- membagi data berdasarkan urutan waktu,
- menyimpan hasilnya ke file terpisah.

Kenapa target biner?

Karena untuk baseline awal, klasifikasi dua kelas jauh lebih mudah dipahami dan sangat relevan untuk IoT operasional.

Kenapa split berbasis waktu?

Karena data IoT adalah time series; model harus belajar dari masa lalu dan diuji pada masa yang lebih baru. Ini lebih realistis dibanding shuffle acak.

In [1]:
import os
from pathlib import Path

while not (Path.cwd() / "scripts" / "mqtt").is_dir():
    os.chdir("..")
print("[INFO] Working directory:", Path.cwd())

[INFO] Working directory: /downloads/iot-ml-book


## 1. Siapkan Dataset

In [2]:
import subprocess, sys
r = subprocess.run([sys.executable, "scripts/prepare_classification_dataset.py"], capture_output=True, text=True)
print(r.stdout)

=== Ringkasan Dataset Klasifikasi ===
Total data      : 2158
Train           : 1510
Validation      : 324
Test            : 324

Distribusi target keseluruhan:
target_abnormal
0    2048
1     110
Name: count, dtype: int64

Distribusi target train:
target_abnormal
0    1436
1      74
Name: count, dtype: int64

File disimpan di: data/processed/classification_room



## 2. Verifikasi File Output

In [3]:
import pandas as pd
out = Path("data/processed/classification_room")
for f in sorted(out.glob("*.csv")):
    d = pd.read_csv(f)
    print(f"{f.name:15s} -> {d.shape}")
X_train = pd.read_csv(out / "X_train.csv")
print(X_train.head())

X_test.csv      -> (324, 17)
X_train.csv     -> (1510, 17)
X_val.csv       -> (324, 17)
full_dataset.csv -> (2158, 22)
test.csv        -> (324, 22)
train.csv       -> (1510, 22)
val.csv         -> (324, 22)
validation.csv  -> (324, 22)
y_test.csv      -> (324, 1)
y_train.csv     -> (1510, 1)
y_val.csv       -> (324, 1)
   temperature_c  humidity_pct  co2_ppm  power_watt  hour  day_of_week  \
0          20.75         49.32      350       95.69     0            3   
1          22.09         52.19      399      119.58     0            3   
2          21.81         48.65      350      112.78     0            3   
3          21.47         48.97      350       95.08     0            3   
4          21.49         49.93      350      101.58     1            3   

   is_weekend  temp_lag_1  hum_lag_1  co2_lag_1  power_lag_1  temp_ma_3  \
0           0       21.60      49.71      350.0       116.53  21.176667   
1           0       20.75      49.32      350.0        95.69  21.480000   
2        

## Jawaban dari data: Hal yang harus diperiksa (Hands-on 20): jawaban dari data

In [4]:
import pandas as pd
d = "data/processed/classification_room/"
tr, va, te = (pd.read_csv(d + f + ".csv") for f in ("train", "val", "test"))
print("Jumlah baris train/val/test:", len(tr), len(va), len(te))
print("Proporsi target_abnormal (train):")
print(tr["target_abnormal"].value_counts(normalize=True).round(3))
print("Urutan waktu dihormati (train < val < test):", tr.timestamp.max() < va.timestamp.min() and va.timestamp.max() < te.timestamp.min())

Jumlah baris train/val/test: 1510 324 324
Proporsi target_abnormal (train):
target_abnormal
0    0.951
1    0.049
Name: proportion, dtype: float64
Urutan waktu dihormati (train < val < test): True


## Panduan Lengkap dari Buku

_Bagian ini dikutip dari buku PDF (sumber kebenaran praktikum) dan diformat ulang agar mudah dibaca. Kode asli buku tidak diubah; bagian instalasi/persiapan lingkungan diringkas karena sudah dijelaskan di README._

### 3. Dataset yang akan dipakai

Kita gunakan dataset sintetis hasil bab sebelumnya, misalnya:

data/synthetic/room_classification_dataset.csv

Kolom umumnya:

- timestamp
- device_id
- temperature_c
- humidity_pct
- co2_ppm
- power_watt
- status_label
- is_anomaly

Bila Anda sudah punya dataset fitur, itu lebih baik. Jika belum, kita akan membuat beberapa fitur ringan di dalam skrip.

### 6. Source code utama

Buat file scripts/prepare_classification_dataset.py:

*(Kode lengkap tersedia pada sel kode notebook ini.)*

Jalankan:

```python
python scripts/prepare_classification_dataset.py
```

### 7. Hal yang harus diperiksa pembaca

Setelah skrip dijalankan, periksa:

- apakah jumlah data train/validation/test masuk akal,
- apakah target abnormal terlalu sedikit atau cukup,
- apakah fitur yang dibuat tidak mengandung kebocoran masa depan,
- apakah semua file berhasil disimpan.

Cek cepat dengan Python

```python
import pandas as pd
train = pd.read_csv("data/processed/classification_room/train.csv")
print(train.head())
print(train["target_abnormal"].value_counts(normalize=True))
```

### 8. Insight penting

Latihan ini mengajarkan bahwa dataset klasifikasi yang baik bukan hanya soal punya label, tetapi juga:

- targetnya jelas,
- fiturnya relevan,
- urutan waktunya dihormati,
- dan distribusi labelnya dipahami.

## Penutup

Split berbasis waktu menjaga realisme IoT: model belajar dari masa lalu, diuji pada masa
yang lebih baru. Dataset ini langsung dipakai pada Hands-on 24 (klasifikasi status
perangkat) dan Hands-on 25 (perbandingan 3 model).